In [1]:
# IMPORTAR LIBRERÍAS NECESARIAS

# numpy: librería matemática por excelencia para manejar los datos numéricos.
import numpy as np

# matplotlib.pyplot: para hacer las gráficas (plots).
import matplotlib.pyplot as plt

# pathlib: para manejar rutas de carpetas y archivos. 
from pathlib import Path

# scipy.io: para abrir archivos de MATLAB
# h5py: para leer archivos .mat v7.3
import scipy.io as sio
from scipy.signal import butter, sosfiltfilt, hilbert, spectrogram
from scipy.interpolate import interp1d

import h5py

# pandas: para manejar datos en forma de tablas (dataframes).
import pandas as pd

In [ ]:
# ==============================================================
# 1. PARÁMETROS GENERALES Y GENERACIÓN AUTOMÁTICA DE METADATOS 
# ==============================================================

# PARÁMETROS DEL REGISTRO
n_channels = 16 # Número de canales del probe
fs = 2500       # Frecuencia de muestreo (Hz)

# DEFINIR CARPETA PRINCIPAL Y GRUPOS DE ESTUDIO
carpeta_principal = Path(r"D:\Violeta\ANALISIS")
carpeta_resultados = carpeta_principal / "ProcesamientoNatalia"
# Crear la carpeta de resultados si no existe
carpeta_resultados.mkdir(parents=True, exist_ok=True)

grupos_estudio = ["WT", "NPC", "EFV"]

# RASTREO DE SESIONES Y CONSTRUCCIÓN DE METADATOS
registros_metadatos = []

# Busca todos los LFP_downsampled.dat dentro de la estructura de carpetas
for dat_file in carpeta_principal.rglob("LFP_downsampled.dat"):
    folder_sesion = dat_file.parent  # Carpeta raíz de la sesión
    
    # Rutas exactas según la estructura 
    mat_ripple = folder_sesion / "analyset" / "rippleAnalysis.mat"
    mat_csd = folder_sesion / "rippleCSDs.mat"
    
    # Solo considera válida la sesión si existe el archivo de análisis de ripples
    if mat_ripple.exists():
        # Identificar el grupo analizando la ruta completa
        ruta_str = str(folder_sesion).upper()
        grupo_detectado = "DESCONOCIDO"
        for grupo in grupos_estudio:
            if grupo in ruta_str:
                grupo_detectado = grupo
                break
        
        # Extraer ID del ratón y de la sesión basándose en la jerarquía de carpetas
        id_raton = folder_sesion.parent.name
        id_sesion = folder_sesion.name
        
        registros_metadatos.append({
            "Grupo": grupo_detectado,
            "Raton": id_raton,
            "Sesion": id_sesion,
            "Ruta_DAT": str(dat_file),
            "Ruta_MAT_Ripple": str(mat_ripple),
            "Ruta_MAT_CSD": str(mat_csd) if mat_csd.exists() else None,
            "Ruta_Carpeta": str(folder_sesion)
        })

# Convertir a DataFrame (Tabla de metadatos)
df_metadatos = pd.DataFrame(registros_metadatos)

# Guardar la tabla en la carpeta ProcesamientoNatalia 
ruta_metadata_csv = carpeta_resultados / "metadata_proyecto.csv"
df_metadatos.to_csv(ruta_metadata_csv, index=False)

print("="*65)
print(f"Total de sesiones válidas encontradas: {len(df_metadatos)}")
if not df_metadatos.empty:
    print("\nDesglose por grupos detectados:")
    print(df_metadatos['Grupo'].value_counts())
print(f"\nMetadatos guardados correctamente en:\n{ruta_metadata_csv}")
print("="*65)


Total de sesiones válidas encontradas: 95

Desglose por grupos detectados:
Grupo
NPC    47
EFV    38
WT     10
Name: count, dtype: int64

Metadatos guardados correctamente en:
D:\Violeta\ANALISIS\ProcesamientoNatalia\metadata_proyecto.csv


In [17]:
# =============================================================
# 2. FUNCIÓN PROCESADORA PRINCIPAL PARA UNA SESIÓN INDIVIDUAL
# =============================================================

def obtener_variable_mat(ruta_file, posibles_claves):
    """
    Busca la variable 'iRipS' (o variantes) tanto en v7 como en v7.3 (HDF5).
    """
    # Intentar con scipy (MATLAB v7 o inferior)
    try:
        mat_data = sio.loadmat(ruta_file)
        for clave in posibles_claves:
            if clave in mat_data:
                return mat_data[clave]
    except NotImplementedError:
        pass  # Es versión v7.3 (HDF5)

    # Intentar con h5py (MATLAB v7.3 / HDF5)
    with h5py.File(ruta_file, 'r') as f:
        for clave in posibles_claves:
            if clave in f:
                datos = np.array(f[clave])
                # Ajustar dimensiones si viene transpuesto [2, N] desde MATLAB a [N, 2]
                if datos.ndim == 2 and datos.shape[0] == 2:
                    datos = datos.T
                return datos
        
        # Si no lo encuentra, lanzar las claves reales del archivo
        raise KeyError(f"No se encontró ninguna clave de {posibles_claves}. Claves reales en el archivo: {list(f.keys())}")

def procesar_una_sesion(fila_metadata):
    """
    Procesa de principio a fin una sesión individual:
    - Carga LFP, ripples y CSD.
    - Filtra y re-alinea las ventanas a +/- 30 ms centradas en el pico absoluto.
    - Calcula métricas (Amplitud, Ripple Power, CSD Medio).
    - Clasifica las capas anatómicas (Or, Pyr, Rad, SLM).
    - Guarda Excel individual y figuras dentro de [Carpeta_Sesion]/ProcesamientoNatalia.
    - Devuelve un resumen con métricas clave para la estadística global.
    """
    
    # CREAR CARPETA DE RESULTADOS INDIVIDUAL EN LA SESIÓN
    carpeta_sesion = Path(fila_metadata['Ruta_Carpeta'])
    carpeta_out_indiv = carpeta_sesion / "ProcesamientoNatalia"
    carpeta_out_indiv.mkdir(parents=True, exist_ok=True)
    
    # CARGA DE ARCHIVOS (.DAT y .MAT)
    ruta_dat = fila_metadata['Ruta_DAT']
    ruta_mat_ripple = fila_metadata['Ruta_MAT_Ripple']
    ruta_mat_csd = fila_metadata['Ruta_MAT_CSD']
    
    # Cargar matriz LFP
    datos_matriz = np.fromfile(ruta_dat, dtype=np.int16).reshape(-1, n_channels)
    puntos_de_tiempo, _ = datos_matriz.shape
    
    # Cargar eventos de ripples 
    posibles_nombres_irips = ['iRipS', 'irips', 'sIrips', 'ripples']
    irips = obtener_variable_mat(ruta_mat_ripple, posibles_nombres_irips)
    
    irips_python = irips.astype(int) - 1  # Base 0
    puntos_ventana = int(0.030 * fs)     # +/- 30 ms
    
    # FILTRADO Y RE-ALINEACIÓN (+/- 30 ms centrados en el pico)
    f_low, f_high = 100, 300
    sos = butter(4, [f_low, f_high], btype='band', fs=fs, output='sos')
    lfp_ripple_band = sosfiltfilt(sos, datos_matriz, axis=0)
    
    # Envolvente de Hilbert y Potencia
    envolvente = np.abs(hilbert(lfp_ripple_band, axis=0))
    potencia_instantanea = envolvente ** 2
    
    lista_ripples = []
    lista_potencia = []
    lista_ripple_band = []
    picos_centrados = []
    
    for inicio, fin in irips_python:
        segmento_orig = lfp_ripple_band[inicio:fin, :]
        if segmento_orig.shape[0] > 0:
            # Canal de mayor amplitud y pico absoluto
            amplitud_por_canal = np.max(np.abs(segmento_orig), axis=0)
            canal_maximo = np.argmax(amplitud_por_canal)
            pico_relativo = np.argmax(np.abs(segmento_orig[:, canal_maximo]))
            idx_pico_global = inicio + pico_relativo
            
            # Nuevos límites
            nuevo_inicio = idx_pico_global - puntos_ventana
            nuevo_fin = idx_pico_global + puntos_ventana + 1
            
            if nuevo_inicio >= 0 and nuevo_fin <= puntos_de_tiempo:
                lista_ripples.append(datos_matriz[nuevo_inicio:nuevo_fin, :])
                lista_potencia.append(potencia_instantanea[nuevo_inicio:nuevo_fin, :])
                lista_ripple_band.append(lfp_ripple_band[nuevo_inicio:nuevo_fin, :])
                picos_centrados.append(idx_pico_global)
                
    matriz_ripples = np.array(lista_ripples)
    matriz_potencia = np.array(lista_potencia)
    matriz_ripple_band = np.array(lista_ripple_band)
    
    # Promedios
    power_medio_por_canal = np.mean(matriz_potencia, axis=(0, 1))
    p2p_por_ripple = np.ptp(matriz_ripples, axis=1)
    promedio_indiv_por_canal = np.mean(p2p_por_ripple, axis=0)
    max_absoluta_por_canal = np.max(p2p_por_ripple, axis=0)
    
    # CARGA Y CÁLCULO DE CSD
    try:
        mat_csd = sio.loadmat(ruta_mat_csd)
        ripple_struct = mat_csd['rippleCSDs'][0, 0]
        tiempo_csd = ripple_struct['t'].flatten()
        csd_raw = ripple_struct['CSDs']
        csd_matrix_full = np.mean(csd_raw, axis=2).T
    except Exception:
        with h5py.File(ruta_mat_csd, 'r') as f:
            csd_group = f['rippleCSDs']
            tiempo_csd = np.array(csd_group['t']).flatten()
            csd_raw = np.array(csd_group['CSDs'])
            # En HDF5 el orden de dimensiones suele invertirse
            csd_matrix_full = np.mean(csd_raw, axis=0)
    
    # Recortar a ventana -30 a +30 ms
    idx_30ms = np.where((tiempo_csd >= -30) & (tiempo_csd <= 30))[0]
    csd_matrix = csd_matrix_full[:, idx_30ms]
    tiempo = tiempo_csd[idx_30ms]
    csd_medio_canal = np.mean(csd_matrix, axis=1)
    
    # CLASIFICACIÓN ANATÓMICA DE CAPAS (Or, Pyr, Rad, SLM)
    n_ch_csd = csd_matrix.shape[0]
    ch_offset_csd = (n_channels - n_ch_csd) // 2
    canales_csd = np.arange(1 + ch_offset_csd, n_ch_csd + 1 + ch_offset_csd)
    
    def clasificar_capas(csd_medio, power_medio, canales_csd):
        n_csd = len(csd_medio)
        ch_max_power = np.argmax(power_medio) + 1
        indices_fuentes = [i for i in range(1, n_csd - 1) 
                           if csd_medio[i] > csd_medio[i-1] and csd_medio[i] > csd_medio[i+1] and csd_medio[i] > 0]
        
        if not indices_fuentes:
            idx_pyr_csd = np.argmax(csd_medio)
        else:
            puntuaciones = [csd_medio[idx] / (1.0 + abs(canales_csd[idx] - ch_max_power)) for idx in indices_fuentes]
            idx_pyr_csd = indices_fuentes[np.argmax(puntuaciones)]
            
        ch_pyr = canales_csd[idx_pyr_csd]
        
        idx_rad = next((i for i in range(idx_pyr_csd + 1, n_csd - 1) if csd_medio[i] < csd_medio[i-1] and csd_medio[i] < csd_medio[i+1]), None)
        ch_rad = canales_csd[idx_rad] if idx_rad is not None else None
        
        idx_slm = next((i for i in range(idx_rad + 1, n_csd - 1) if csd_medio[i] > csd_medio[i-1] and csd_medio[i] > csd_medio[i+1]), None) if idx_rad else None
        ch_slm = canales_csd[idx_slm] if idx_slm is not None else None
        
        idx_or = next((i for i in range(idx_pyr_csd - 1, 0, -1) if csd_medio[i] < csd_medio[i+1] and csd_medio[i] < csd_medio[i-1]), None)
        ch_or = canales_csd[idx_or] if idx_or is not None else None
        
        return ch_pyr, ch_rad, ch_slm, ch_or

    ch_pyr, ch_rad, ch_slm, ch_or = clasificar_capas(csd_medio_canal, power_medio_por_canal, canales_csd)
    
    # Diccionario mapa canal -> capa para exportación fácil
    mapa_capas = {ch_pyr: 'Pyr', ch_rad: 'Rad', ch_slm: 'SLM', ch_or: 'Or'}
    
    # GUARDAR EXCEL INDIVIDUAL CON DETALLE DE LOS 16 CANALES
    df_canales_indiv = pd.DataFrame({
        'Canal': np.arange(1, n_channels + 1),
        'Capa_Anatomica': [mapa_capas.get(ch, 'N/A') for ch in range(1, n_channels + 1)],
        'Amp_P2P_Media_uV': promedio_indiv_por_canal,
        'Amp_P2P_Max_uV': max_absoluta_por_canal,
        'Ripple_Power_Medio_uV2': power_medio_por_canal,
    })
    
    # Añadimos el CSD en los canales correspondientes
    csd_columna = [np.nan] * n_channels
    for idx, ch in enumerate(canales_csd):
        csd_columna[ch - 1] = csd_medio_canal[idx]
    df_canales_indiv['CSD_Medio'] = csd_columna
    
    ruta_excel_indiv = carpeta_out_indiv / "resultados_sesion.xlsx"
    df_canales_indiv.to_excel(ruta_excel_indiv, index=False)
    
    # GENERACIÓN Y GUARDADO AUTOMÁTICO DE FIGURA INTEGRADA
    fig, axes = plt.subplots(1, 4, figsize=(18, 6), sharey=True)
    fig.suptitle(f"Sesión: {fila_metadata['Raton']} | {fila_metadata['Sesion']} ({fila_metadata['Grupo']})", fontsize=14, fontweight='bold')
    
    canales = np.arange(1, n_channels + 1)
    
    # Subplot 1: Mapa CSD Espacio-Temporal
    vmax = np.max(np.abs(csd_matrix))
    im = axes[0].imshow(csd_matrix, aspect='auto', cmap='jet', vmin=-vmax, vmax=vmax,
                        extent=[tiempo[0], tiempo[-1], canales_csd[-1] + 0.5, canales_csd[0] - 0.5])
    axes[0].set_title("CSD Espacio-Temporal")
    axes[0].set_xlabel("Tiempo (ms)")
    axes[0].set_ylabel("Canales")
    fig.colorbar(im, ax=axes[0], label="CSD")
    
    # Subplot 2: Perfil CSD Medio
    axes[1].plot(csd_medio_canal, canales_csd, 'k-o', linewidth=2)
    axes[1].axvline(0, color='gray', linestyle='--')
    axes[1].set_title("Perfil CSD Medio")
    axes[1].set_xlabel(r"CSD ($\mu V$)")
    
    # Subplot 3: Amplitud P2P Promedio
    axes[2].plot(promedio_indiv_por_canal, canales, 'b-s', linewidth=2)
    axes[2].set_title("Amplitud P2P Media")
    axes[2].set_xlabel(r"Amplitud ($\mu V$)")
    
    # Subplot 4: Ripple Power Medio
    axes[3].plot(power_medio_por_canal, canales, 'r-^', linewidth=2)
    axes[3].set_title("Ripple Power Medio")
    axes[3].set_xlabel(r"Potencia ($\mu V^2$)")
    
    # Marcar la capa Pyr detectada
    for ax in axes:
        ax.axhline(ch_pyr, color='red', linestyle=':', label='Pyr' if ax == axes[0] else "")
        ax.invert_yaxis()
        
    plt.tight_layout()
    ruta_figura_indiv = carpeta_out_indiv / "figura_integrada.png"
    plt.savefig(ruta_figura_indiv, dpi=200, bbox_inches='tight')
    plt.close(fig)  # Cerrar la figura para liberar memoria RAM
    
    # 8. RESUMEN DICCIONARIO PARA RESULTADOS GLOBALES
    return {
        "Grupo": fila_metadata['Grupo'],
        "Raton": fila_metadata['Raton'],
        "Sesion": fila_metadata['Sesion'],
        "Num_Ripples": len(matriz_ripples),
        "Canal_Pyr": ch_pyr,
        "Canal_Rad": ch_rad,
        "Canal_SLM": ch_slm,
        "Canal_Or": ch_or,
        "Pyr_Ripple_Power": power_medio_por_canal[ch_pyr - 1],
        "Pyr_Amp_P2P_Media": promedio_indiv_por_canal[ch_pyr - 1],
        "Max_Ripple_Power_Global": np.max(power_medio_por_canal),
        "Canal_Max_Power": np.argmax(power_medio_por_canal) + 1,
        "QC_Status": "OK"
    }

print("Función 'procesar_una_sesion' definida y lista para usar.")

Función 'procesar_una_sesion' definida y lista para usar.


In [18]:
# =========================================================
# 3. BUCLE BATCH Y CONSOLIDACIÓN DE RESULTADOS GLOBALES
# =========================================================

# CARGAR LA TABLA DE METADATOS
ruta_metadata = carpeta_resultados / "metadata_proyecto.csv"
df_meta = pd.read_csv(ruta_metadata)

resultados_globales = []
log_qc = []

print("="*65)
print(f"INICIANDO PROCESAMIENTO POR LOTES DE {len(df_meta)} SESIONES...")
print("="*65)

# RECORRER CADA SESIÓN
for idx, fila in df_meta.iterrows():
    nombre_sesion_str = f"{fila['Grupo']} | Ratón: {fila['Raton']} | Sesión: {fila['Sesion']}"
    print(f"\n[Sesión {idx + 1}/{len(df_meta)}] Procesando: {nombre_sesion_str}...")
    
    try:
        # Ejecutar la función procesadora
        res_sesion = procesar_una_sesion(fila)
        resultados_globales.append(res_sesion)
        
        log_qc.append({
            "Grupo": fila["Grupo"],
            "Raton": fila["Raton"],
            "Sesion": fila["Sesion"],
            "Status": "OK",
            "Mensaje": "Procesado correctamente"
        })
        print(f"  -> OK: Generado Excel individual y extraídas métricas (Pyr: Ch{res_sesion['Canal_Pyr']})")
        
    except Exception as e:
        mensaje_error = str(e)
        print(f"  -> ERROR en {nombre_sesion_str}: {mensaje_error}")
        
        log_qc.append({
            "Grupo": fila["Grupo"],
            "Raton": fila["Raton"],
            "Sesion": fila["Sesion"],
            "Status": "ERROR",
            "Mensaje": mensaje_error
        })

# CONSOLIDAR Y GUARDAR TABLAS GLOBALES
df_resultados = pd.DataFrame(resultados_globales)
df_qc = pd.DataFrame(log_qc)

# Rutas de salida en ANALISIS/ProcesamientoNatalia
ruta_excel_global = carpeta_resultados / "Resultados_Globales.xlsx"
ruta_csv_global = carpeta_resultados / "Resultados_Globales.csv"
ruta_qc_log = carpeta_resultados / "QC_Log.csv"

# Exportar a disco
df_resultados.to_excel(ruta_excel_global, index=False)
df_resultados.to_csv(ruta_csv_global, index=False)
df_qc.to_csv(ruta_qc_log, index=False)

print("\n" + "="*65)
print("¡PROCESAMIENTO COMPLETO FINALIZADO!")
print("="*65)
print(f"Sesiones procesadas con éxito: {len(df_resultados)} / {len(df_meta)}")
print(f"Resultados globales guardados en:\n  -> {ruta_excel_global}")
print(f"Log de Control de Calidad (QC) guardado en:\n  -> {ruta_qc_log}")
print("="*65)

# Mostrar un resumen de la tabla global obtenida
df_resultados.head()

INICIANDO PROCESAMIENTO POR LOTES DE 95 SESIONES...

[Sesión 1/95] Procesando: WT | Ratón: WT_90 | Sesión: 2025_07_16_0000...
  -> ERROR en WT | Ratón: WT_90 | Sesión: 2025_07_16_0000: "No se encontró ninguna clave de ['iRipS', 'irips', 'sIrips', 'ripples']. Claves reales en el archivo: ['#refs#', '#subsystem#', 'rippleAnalysis']"

[Sesión 2/95] Procesando: WT | Ratón: WT_90 | Sesión: 2025_07_16_0001...
  -> ERROR en WT | Ratón: WT_90 | Sesión: 2025_07_16_0001: "No se encontró ninguna clave de ['iRipS', 'irips', 'sIrips', 'ripples']. Claves reales en el archivo: ['#refs#', '#subsystem#', 'rippleAnalysis']"

[Sesión 3/95] Procesando: WT | Ratón: WT_90 | Sesión: 2025_07_17_0000...
  -> ERROR en WT | Ratón: WT_90 | Sesión: 2025_07_17_0000: "No se encontró ninguna clave de ['iRipS', 'irips', 'sIrips', 'ripples']. Claves reales en el archivo: ['#refs#', '#subsystem#', 'rippleAnalysis']"

[Sesión 4/95] Procesando: WT | Ratón: WT_90 | Sesión: 2025_07_17_0001...
  -> ERROR en WT | Ratón: WT_90

""
